In [ ]:
import warnings, numpy as np
from pyspark.sql import SparkSession, Window, functions as F

warnings.filterwarnings("ignore")


class ClusterFeatureEngineering:
    def __init__(self):
        self.panel_with_cluster_path = "/user/data/kshape/clustering"
        self.full_panel_path = "/user/data/kshape/preprocess/full_panel"
        self.cluster_csv_path = "/workspace/code/kshape/clustering/assignments.csv"
        self.output_path = "/user/data/kshape/feature_engineering"
        self.bins_per_day, self.missing_cluster_id = 48, -1
        self.spark = (
            SparkSession.builder.appName("FeatureEngineering")
            .config("spark.sql.session.timeZone", "America/New_York")
            .config("spark.sql.adaptive.enabled", "true")
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
            .config("spark.sql.adaptive.skewJoin.enabled", "true")
            .config("spark.sql.shuffle.partitions", "64")
            .config("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
            .config("spark.sql.debug.maxToStringFields", "500")
            .config("spark.sql.maxPlanStringLength", "200000")
            .config("spark.master", "yarn")
            .config("spark.submit.deployMode", "client")
            .config("spark.driver.host", "192.168.1.111")
            .config("spark.driver.bindAddress", "0.0.0.0")
            .getOrCreate()
        )
        self.spark.sparkContext.setLogLevel("WARN")
        jvm = self.spark._jvm
        self.fs = jvm.org.apache.hadoop.fs.FileSystem.get(self.spark._jsc.hadoopConfiguration())
        self.Path = jvm.org.apache.hadoop.fs.Path

    def exists(self, path):
        return self.fs.exists(self.Path(path))

    @staticmethod
    def z(x):
        m, s = np.nanmean(x), np.nanstd(x)
        return (x - m) / (s if s >= 1e-12 and not np.isnan(s) else 1.0)

    def run(self):
        panel = self.spark.read.parquet(self.panel_with_cluster_path if self.exists(self.panel_with_cluster_path) else self.full_panel_path)
        if "cluster_id" not in panel.columns:
            if not self.exists(self.cluster_csv_path):
                raise ValueError(f"Không tìm thấy cluster_id trong panel và cũng không thấy file: {self.cluster_csv_path}")
            assign = (
                self.spark.read.csv(self.cluster_csv_path, header=True, inferSchema=True)
                .dropna(subset=["PULocationID", "cluster_id"])
                .select(F.col("PULocationID").cast("int"), F.col("cluster_id").cast("int"))
            )
            panel = panel.join(F.broadcast(assign), on="PULocationID", how="left")

        panel = (
            panel.withColumn("cluster_id", F.coalesce(F.col("cluster_id"), F.lit(self.missing_cluster_id)).cast("int"))
            .filter(F.col("pickup_bin_30m").isNotNull() & F.col("PULocationID").isNotNull())
            .withColumn("pickup_demand", F.coalesce(F.col("pickup_demand"), F.lit(0.0)).cast("double"))
        )
        if "half_hour_slot" not in panel.columns and "slot_30m" in panel.columns:
            panel = panel.withColumn("half_hour_slot", F.col("slot_30m").cast("int"))
        if "year" not in panel.columns:
            panel = panel.withColumn("year", F.year("pickup_bin_30m").cast("int"))
        if "ready" in panel.columns:
            panel = panel.withColumn("ready", F.coalesce(F.col("ready"), F.lit(0)).cast("int"))
        panel = panel.persist()

        cluster = (
            panel.filter(F.col("cluster_id") != self.missing_cluster_id)
            .groupBy("cluster_id", "pickup_bin_30m")
            .agg(F.avg("pickup_demand").alias("cluster_avg_demand_t"))
        )
        w = Window.partitionBy("cluster_id").orderBy("pickup_bin_30m")
        w24 = w.rowsBetween(-self.bins_per_day, -1)
        for i in range(1, 6):
            cluster = cluster.withColumn(f"cluster_demand_t_{i}", F.lag("cluster_avg_demand_t", i).over(w))
        cluster = (
            cluster.withColumn("cluster_rolling_obs_24h", F.count("cluster_avg_demand_t").over(w24))
            .withColumn("cluster_rolling_max_24h", F.max("cluster_avg_demand_t").over(w24))
            .withColumn("cluster_rolling_min_24h", F.min("cluster_avg_demand_t").over(w24))
            .withColumn("cluster_rolling_mean_24h", F.avg("cluster_avg_demand_t").over(w24))
            .withColumn("cluster_rolling_std_24h", F.stddev("cluster_avg_demand_t").over(w24))
        )
        panel = panel.join(F.broadcast(cluster), on=["cluster_id", "pickup_bin_30m"], how="left")

        panel = panel.fillna({
            **{f"cluster_demand_t_{i}": 0.0 for i in range(1, 6)},
            "cluster_rolling_obs_24h": 0,
            "cluster_rolling_max_24h": 0.0,
            "cluster_rolling_min_24h": 0.0,
            "cluster_rolling_mean_24h": 0.0,
            "cluster_rolling_std_24h": 0.0,
        })
        panel = (
            panel.withColumn("cluster_Diff_t_1", (F.coalesce(F.col("demand_t_1"), F.lit(0.0)) - F.col("cluster_demand_t_1")).cast("double"))
            .withColumn("cluster_Mean_diff_24h", (F.coalesce(F.col("rolling_mean_24h"), F.lit(0.0)) - F.col("cluster_rolling_mean_24h")).cast("double"))
            .withColumn("cluster_Std_diff_24h", (F.coalesce(F.col("rolling_std_24h"), F.lit(0.0)) - F.col("cluster_rolling_std_24h")).cast("double"))
            .withColumn("has_valid_cluster_feature", ((F.col("cluster_id") != self.missing_cluster_id) & F.col("cluster_demand_t_1").isNotNull()).cast("int"))
        )

        # rows = (
        #     panel.filter(F.col("dataset_split") == "train")
        #     .groupBy("PULocationID", "cluster_id")
        #     .agg(F.array_sort(F.collect_list(F.struct(F.col("pickup_bin_30m").alias("pickup_bin_30m"), F.col("pickup_demand").alias("pickup_demand")))).alias("series"))
        #     .collect()
        # )
        # if rows:
        #     loc_ids = [int(r["PULocationID"]) for r in rows]
        #     loc_clusters = [int(r["cluster_id"]) for r in rows]
        #     Xz = np.array([self.z(np.array([float(x["pickup_demand"] or 0.0) for x in r["series"]], dtype=np.float64)) for r in rows])
        #     reps = {cid: self.z(Xz[[i for i, c in enumerate(loc_clusters) if c == cid]].mean(axis=0)) for cid in sorted(set(loc_clusters))}
        #     sims = [{
        #         "PULocationID": loc_id,
        #         "intra_cluster_similarity": float(np.mean(x * reps[cid])),
        #         "inter_cluster_similarity": max([float(np.mean(x * rep)) for other_cid, rep in reps.items() if other_cid != cid] or [0.0]),
        #     } for loc_id, cid, x in zip(loc_ids, loc_clusters, Xz)]
        #     panel = panel.join(F.broadcast(self.spark.createDataFrame(sims)), on="PULocationID", how="left")

        # --- BẮT ĐẦU ĐOẠN MÃ THAY THẾ ---
        
        # 1. Lọc tập train và loại bỏ missing_cluster_id
        train_df = panel.filter(
            (F.col("dataset_split") == "train") & 
            (F.col("cluster_id") != self.missing_cluster_id)
        )

        # 2. Tính Z-score (self.z) cho pickup_demand của từng Location
        w_loc = Window.partitionBy("PULocationID")
        loc_stats = train_df.withColumn("loc_mean", F.avg("pickup_demand").over(w_loc)) \
                            .withColumn("loc_std", F.stddev("pickup_demand").over(w_loc))
        
        loc_z_df = loc_stats.withColumn(
            "loc_z",
            (F.col("pickup_demand") - F.col("loc_mean")) / F.when(F.col("loc_std") >= 1e-12, F.col("loc_std")).otherwise(1.0)
        ).select("PULocationID", "cluster_id", "pickup_bin_30m", "loc_z")

        # 3. Tính Vector đại diện (reps) cho từng Cluster và chuẩn hóa Z-score
        # Tính trung bình loc_z theo cluster_id và thời gian
        cluster_time_df = loc_z_df.groupBy("cluster_id", "pickup_bin_30m") \
                                  .agg(F.avg("loc_z").alias("avg_loc_z"))

        w_cluster = Window.partitionBy("cluster_id")
        cluster_stats = cluster_time_df.withColumn("rep_mean", F.avg("avg_loc_z").over(w_cluster)) \
                                       .withColumn("rep_std", F.stddev("avg_loc_z").over(w_cluster))

        rep_z_df = cluster_stats.withColumn(
            "rep_z",
            (F.col("avg_loc_z") - F.col("rep_mean")) / F.when(F.col("rep_std") >= 1e-12, F.col("rep_std")).otherwise(1.0)
        ).select(F.col("cluster_id").alias("target_cluster_id"), "pickup_bin_30m", "rep_z")

        # 4. Tính toán độ tương đồng (Similarity = np.mean(x * rep))
        # Nối vector của location với vector của TẤT CẢ các cụm
        joined_df = loc_z_df.join(rep_z_df, on="pickup_bin_30m")

        sim_df = joined_df.withColumn("sim_product", F.col("loc_z") * F.col("rep_z")) \
                          .groupBy("PULocationID", "cluster_id", "target_cluster_id") \
                          .agg(F.avg("sim_product").alias("similarity"))

        # 5. Tách thành intra_cluster (cùng cụm) và inter_cluster (khác cụm)
        intra_df = sim_df.filter(F.col("cluster_id") == F.col("target_cluster_id")) \
                         .select("PULocationID", F.col("similarity").alias("intra_cluster_similarity"))

        inter_df = sim_df.filter(F.col("cluster_id") != F.col("target_cluster_id")) \
                         .groupBy("PULocationID") \
                         .agg(F.max("similarity").alias("inter_cluster_similarity"))

        # Tổng hợp kết quả và nối ngược lại vào panel gốc
        sims_final = intra_df.join(inter_df, on="PULocationID", how="outer")
        
        # Nếu có dữ liệu train thì mới join (tránh lỗi khi chạy test nhỏ)
        panel = panel.join(sims_final, on="PULocationID", how="left")
        panel = panel.fillna({"intra_cluster_similarity": 0.0, "inter_cluster_similarity": 0.0})

        cols = [
            *"pickup_bin_30m date dataset_split PULocationID pickup_demand hour minute slot_30m half_hour_slot day_of_week week_of_year month year day_of_month is_weekday is_weekend ha_output ewma_output history_bins_available enough_history_lag5 enough_history_24h enough_history_ewma enough_history_1w is_cold_start ready cluster_id cluster_rolling_obs_24h cluster_rolling_max_24h cluster_rolling_min_24h cluster_rolling_mean_24h cluster_rolling_std_24h cluster_Diff_t_1 cluster_Mean_diff_24h cluster_Std_diff_24h intra_cluster_similarity inter_cluster_similarity has_valid_cluster_feature".split(),
            *[f"demand_t_{i}" for i in range(1, 6)],
            *"rolling_obs_24h rolling_max_24h rolling_min_24h rolling_mean_24h rolling_std_24h rolling_skew_24h".split(),
            *[f"cluster_demand_t_{i}" for i in range(1, 6)],
        ]
        out = panel.select(*[c for c in cols if c in panel.columns])
        writer = out.write.mode("overwrite")
        parts = [c for c in ["dataset_split", "year", "month"] if c in out.columns]
        if parts:
            writer = writer.partitionBy(*parts)
        writer.parquet(self.output_path)

        print("=" * 120)
        print("STEP 4 — FEATURE ENGINEERING DONE")
        print("=" * 120)
        print(f"input: {self.panel_with_cluster_path if self.exists(self.panel_with_cluster_path) else self.full_panel_path}")
        print(f"output: {self.output_path}")
        print(f"columns: {len(out.columns)}")


ClusterFeatureEngineering().run()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-96564c0f-057e-4d04-9c32-5562cbf84708;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 135ms :: artifacts dl 5ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

STEP 4 — FEATURE ENGINEERING DONE
input: /user/data/clustering
output: /user/data/feature_engineering
columns: 53
